[< Back to Main README](../README.md) | [Demo README](./README.md)

# RAG vs Graph-RAG: Reducing Agent Hallucinations

**Research Validated:**
- [Internal Representations as Indicators of Hallucinations](https://arxiv.org/pdf/2601.05214)
- [RAG-KG-IL: Multi-Agent Hybrid Framework](https://arxiv.org/pdf/2503.13514)
- [MetaRAG: Metamorphic Testing for Hallucination Detection](https://arxiv.org/pdf/2509.09360)

---

## What We're Testing

| Test | What It Measures | RAG Expected | Graph-RAG Expected |
|------|-----------------|--------------|--------------------|
| Aggregation | Can it compute averages? | ❌ Guesses | ✅ Native AVG() |
| Counting | Can it count across docs? | ❌ Can't | ✅ Native COUNT() |
| Multi-hop | Can it traverse relations? | ❌ Limited | ✅ Cypher traversal |
| Out-of-domain | Does it hallucinate? | ❌ Fabricates | ✅ Honest failure |

---

## Configure AWS Credentials

This demo uses Amazon Bedrock (default model provider for Strands Agents). Ensure your AWS credentials are configured.

To use a different provider, see the [Model Providers documentation](https://strandsagents.com/docs/user-guide/concepts/model-providers/amazon-bedrock/).

In [1]:
# Ensure AWS region is set (required for Bedrock in Workshop Studio)
import os
if not os.environ.get("AWS_DEFAULT_REGION") and not os.environ.get("AWS_REGION"):
    os.environ["AWS_DEFAULT_REGION"] = "us-east-1"

import os
# Verify AWS credentials are available
import boto3
sts = boto3.client("sts")
identity = sts.get_caller_identity()
print(f"AWS Account: {identity['Account'][-4:].rjust(12, '*')}")  # Show last 4 digits only
print(f"Region: {boto3.session.Session().region_name}")
print("\u2705 AWS credentials configured")

# To use OpenAI instead of Bedrock:
#   pip install "strands-agents[openai]"
#   os.environ["OPENAI_API_KEY"] = "your-key-here"
#   from strands.models.openai import OpenAIModel
#   MODEL = OpenAIModel(model_id="gpt-4o-mini")


AWS Account: ********1974
Region: us-east-1
✅ AWS credentials configured


## Setup

In [2]:
import os
os.environ['OTEL_SDK_DISABLED'] = 'true'
from dotenv import load_dotenv
load_dotenv()

from strands import Agent, tool
from neo4j import GraphDatabase
import faiss
import json
import boto3
import numpy as np

# Get Neo4j credentials from environment variables
# Workshop Studio: Pre-configured in /etc/environment by CloudFormation
# Self-paced: Set NEO4J_URI, NEO4J_USERNAME and NEO4J_PASSWORD before running
NEO4J_URI = os.getenv('NEO4J_URI')
NEO4J_USERNAME = os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')

if not NEO4J_URI or not NEO4J_USERNAME or not NEO4J_PASSWORD:
    raise ValueError(
        'Neo4j credentials not found.\n'
        'Expected environment variables: NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD\n'
        'Workshop Studio: Check that CloudFormation deployment completed successfully.\n'
        'Self-paced: Set these variables before running the notebook.'
    )

print(f"✅ Neo4j URI: {NEO4J_URI}")
print(f"✅ Neo4j password: {'*' * len(NEO4J_PASSWORD)} ({len(NEO4J_PASSWORD)} chars)")

# Amazon Bedrock Nova 2 for embeddings (no SentenceTransformer / OpenAI needed)
_bedrock = boto3.client("bedrock-runtime", region_name="us-east-1")

def _embed(text):
    """Embed text using Amazon Bedrock Nova 2 Multimodal Embeddings."""
    resp = _bedrock.invoke_model(
        modelId="amazon.nova-2-multimodal-embeddings-v1:0",
        body=json.dumps({
            "taskType": "SINGLE_EMBEDDING",
            "singleEmbeddingParams": {
                "embeddingPurpose": "GENERIC_INDEX",
                "embeddingDimension": 1024,
                "text": {"truncationMode": "END", "value": text[:8000]},
            },
        }),
        contentType="application/json",
        accept="application/json",
    )
    result = json.loads(resp["body"].read())
    return np.array([result["embeddings"][0]["embedding"]], dtype="float32")

# Build FAISS index if not already built
import subprocess, sys, os
from pathlib import Path

def build_faiss_if_needed():
    full_index = "faqs_vector.index"
    full_docs = "faqs_docs.json"
    
    if os.path.exists(full_index) and os.path.exists(full_docs):
        return full_index, full_docs
    
    # Extract hotel FAQ data if not already extracted
    data_dir = "data"
    if not os.path.exists(data_dir) or not list(Path(data_dir).glob("*.txt")):
        zip_file = "hotel-faqs.zip"
        if not os.path.exists(zip_file):
            raise FileNotFoundError(
                f"hotel-faqs.zip not found in {os.getcwd()}.\n"
                "At an AWS Event: the data is pre-loaded — check that you cd to the right folder.\n"
                "Self-paced: download hotel-faqs.zip from the workshop releases page."
            )
        print(f"Extracting {zip_file}...")
        import zipfile
        with zipfile.ZipFile(zip_file, "r") as zf:
            zf.extractall(data_dir)
        print(f"Extracted {len(list(Path(data_dir).glob('*.txt')))} hotel FAQ documents.")
    
    print("Building FAISS index from hotel FAQ data (full: 300 docs, ~10 min)...")
    result = subprocess.run([sys.executable, "load_vector_data.py"], capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"Failed to build FAISS index:\n{result.stderr}")
    print("FAISS index built.")
    return full_index, full_docs

index_file, docs_file = build_faiss_if_needed()

# Load FAISS
index = faiss.read_index(index_file)
with open(docs_file, "r", encoding="utf-8") as f:
    documents = json.load(f)
print(f"✅ FAISS: {len(documents)} documents")

# Check Neo4j
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
with driver.session() as session:
    count = session.run('MATCH (h:Hotel) RETURN count(h) as c').single()['c']
    print(f"✅ Neo4j: {count} hotels in knowledge graph")
driver.close()

✅ Neo4j URI: neo4j+s://f024ea61.databases.neo4j.io
✅ Neo4j password: ******************************************* (43 chars)
✅ FAISS: 300 documents


✅ Neo4j: 30 hotels in knowledge graph


## Define Tools & Agents

In [3]:
# Tool observability hook - captures tool inputs and outputs
from strands.hooks import HookProvider, HookRegistry
from strands.hooks.events import BeforeToolCallEvent, AfterToolCallEvent

class ToolObservabilityHook(HookProvider):
    """Display tool inputs and outputs for debugging and comparison.
    
    Uses Strands hook system to intercept tool execution:
    - BeforeToolCallEvent: captures tool name and input parameters
    - AfterToolCallEvent: captures tool result and status
    
    This approach keeps tools clean (no logging code inside them)
    and provides a reusable observability layer.
    """
    
    def register_hooks(self, registry: HookRegistry) -> None:
        registry.add_callback(BeforeToolCallEvent, self.on_before_tool)
        registry.add_callback(AfterToolCallEvent, self.on_after_tool)
    
    def on_before_tool(self, event: BeforeToolCallEvent) -> None:
        """Called before tool execution - shows input parameters."""
        tool_input = event.tool_use.get("input", {})
        
        print(f"   📥 Tool Input:")
        for key, value in tool_input.items():
            val_str = str(value)
            # Truncate long values (e.g., Cypher queries)
            if len(val_str) > 300:
                val_str = val_str[:300] + "..."
            print(f"      {key}: {val_str}")
    
    def on_after_tool(self, event: AfterToolCallEvent) -> None:
        """Called after tool execution - shows output/result."""
        status = event.result["status"]
        
        # Handle errors
        if status == "error" or event.exception:
            print(f"   ❌ Error: {event.exception}")
            return
        
        # Extract content from result
        if event.result["content"]:
            for content_item in event.result["content"]:
                # Most common: text output
                if "text" in content_item:
                    result_text = content_item["text"]
                    # Show first 800 chars of output
                    if len(result_text) > 800:
                        print(f"   📤 Tool Output (truncated):\n{result_text[:800]}...")
                    else:
                        print(f"   📤 Tool Output:\n{result_text}")
                # Handle JSON output
                elif "json" in content_item:
                    print(f"   📤 Tool Output (JSON):\n{content_item['json']}")

print("✅ Tool observability hook defined")

✅ Tool observability hook defined


In [4]:
@tool
def search_faqs(query: str) -> str:
    """Search hotel FAQs using vector similarity (Traditional RAG)."""
    query_embedding = _embed(query)
    distances, indices = index.search(query_embedding, 3)
    results = []
    for idx in indices[0]:
        doc = documents[idx]
        results.append(f"[{doc['filename']}]\n{doc['text'][:500]}...")
    return "\n\n".join(results)

@tool
def query_knowledge_graph(cypher_query: str) -> str:
    """Execute a Cypher query against the hotel knowledge graph.
    
    Node labels: Hotel, Room, Amenity, Policy, Service
    Hotel properties: name, address, guest_rating, total_rooms, email, phone
    Room properties: type, bed_configuration, max_occupancy, min_rate, max_rate
    Amenity properties: name, description, fee
    Policy properties: name, description
    Service properties: name, description, cost, hours, is_available, is_complimentary
    
    Relationships: (Hotel)-[:HAS_ROOM]->(Room), (Hotel)-[:OFFERS_AMENITY]->(Amenity),
                   (Hotel)-[:HAS_POLICY]->(Policy), (Hotel)-[:PROVIDES_SERVICE]->(Service)
    
    Location is in Hotel.address property. Use: WHERE h.address CONTAINS 'Cairo'
    IMPORTANT: All property names use snake_case (e.g., guest_rating NOT guestRating)
    """
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
    with driver.session() as session:
        try:
            result = session.run(cypher_query)
            records = list(result)
            if not records:
                return "No results found."
            output = f"Found {len(records)} results:\n"
            for record in records[:15]:
                output += f"  {dict(record.items())}\n"
            return output
        except Exception as e:
            return f"Query error: {str(e)}"
        finally:
            driver.close()

# Model configuration — Amazon Bedrock (default, no extra import needed)
# Strands Agents uses Bedrock by default when no model is specified.
#
# To use OpenAI instead:
#   pip install "strands-agents[openai]"
#   from strands.models.openai import OpenAIModel
#   MODEL = OpenAIModel(model_id="gpt-4o-mini")
#   rag_agent = Agent(..., model=MODEL)
#
# See all providers: https://strandsagents.com/docs/user-guide/concepts/model-providers/

# Create observability hook instance
observability_hook = ToolObservabilityHook()

rag_agent = Agent(
    name="RAG_Agent",
    system_prompt="You are a travel agent. Use vector search to find relevant FAQ information.",
    tools=[search_faqs],
    hooks=[observability_hook],
)

graph_agent = Agent(
    name="GraphRAG_Agent",
    system_prompt="You are a travel agent. Use the knowledge base to answer questions accurately. You can run multiple queries.",
    tools=[query_knowledge_graph],
    hooks=[observability_hook],
)

print("✅ Agents ready")

✅ Agents ready


---

## Test 1: Aggregation

**Paper:** "RAG cannot compute aggregations — LLM guesses from text chunks"

**Query:** What is the average guest rating across all hotels in Paris?

### Token Counting with Strands

Strands Agents provides **native token counting** through `AgentResult.metrics`. No custom helper functions needed:

```python
result = agent("Your query")
usage = result.metrics.accumulated_usage
print(f"Tokens: {usage['inputTokens']} in, {usage['outputTokens']} out, {usage['totalTokens']} total")
```

Available metrics:
- `inputTokens` — Tokens sent to the model
- `outputTokens` — Tokens generated by the model  
- `totalTokens` — Total (input + output)
- `cacheReadInputTokens` — Tokens read from cache (when using prompt caching)
- `cacheWriteInputTokens` — Tokens written to cache (when using prompt caching)

Token counts shown below each agent response demonstrate cost differences between approaches.

In [5]:
query = "What is the average guest rating of all hotels in Paris?"
print(f"👤 Query: {query}\n")

print("[TRADITIONAL RAG]")
r = rag_agent(query)

if r.metrics:
    usage = r.metrics.accumulated_usage
    print(f"\n💰 Tokens: {usage['inputTokens']} in, {usage['outputTokens']} out, {usage['totalTokens']} total")

👤 Query: What is the average guest rating of all hotels in Paris?

[TRADITIONAL RAG]


Let me search the FAQs for information about hotel ratings in Paris!


Tool #1: search_faqs


   📥 Tool Input:
      query: average guest rating hotels Paris


   📤 Tool Output (truncated):
[hotel-paris-001.txt]
# AnyCompany Paris Champs-Élysées

## Hotel Overview
AnyCompany Paris Champs-Élysées is located in Paris, France. Luxury on the world's most famous avenue

**Guest Rating:** 4.9/5.0
**Total Rooms:** 180

## Contact Information
**Address:** 789 Avenue des Champs-Élysées, 75008 Paris
**Phone:** +33-1-5555-0001
**Email:** paris@anycompany.com

## Check-in & Check-out

### Check-in Time
Standard check-in time is 3:00 PM. Early check-in is subject to availability and may be requested at the time of b...

[hotel-paris-002.txt]
# AnyCompany Le Marais

## Hotel Overview
AnyCompany Le Marais is located in Paris, France. Historic charm in the trendy Marais district

**Guest Rating:** 4.5/5.0
**Total Rooms:** 120

## Contact Information
**Address:** 321 Rue de Rivoli, 75004 Pari...


...

-Élysées** leads with an impressive **4.9/

5.0**, while the **AnyCompany Le Marais** follows closely with a strong

 **4.5/5.0**. Would you like more details

 about either of these hotels or help with a booking?


💰 Tokens: 1796 in, 286 out, 2082 total


In [6]:
query = "What is the average guest rating of all hotels in Paris?"

print("[GRAPH-RAG]")
r = graph_agent(query)

if r.metrics:
    usage = r.metrics.accumulated_usage
    print(f"\n💰 Tokens: {usage['inputTokens']} in, {usage['outputTokens']} out, {usage['totalTokens']} total")

print("\n📊 RAG guesses from top 3 docs | Graph-RAG calculates exact AVG() across all Paris hotels")

[GRAPH-RAG]



Tool #1: query_knowledge_graph


   📥 Tool Input:
      cypher_query: MATCH (h:Hotel) WHERE h.address CONTAINS 'Paris' RETURN avg(h.guest_rating) AS average_guest_rating


   📤 Tool Output:
Found 1 results:
  {'average_guest_rating': 4.7}



The **average guest rating** of all hotels in Paris is **4.7 out of 5**. 

...

 regarded by guests! Would

 you like to explore specific hotels in Paris, check available

 rooms, or get more details about amenities and services? I'm

 happy to help! 🗼


💰 Tokens: 1798 in, 165 out, 1963 total

📊 RAG guesses from top 3 docs | Graph-RAG calculates exact AVG() across all Paris hotels


In [7]:
query = "How many hotels in the database have a swimming pool?"
print(f"👤 Query: {query}\n")

print("[TRADITIONAL RAG]")
r = rag_agent(query)

if r.metrics:
    usage = r.metrics.accumulated_usage
    print(f"\n💰 Tokens: {usage['inputTokens']} in, {usage['outputTokens']} out, {usage['totalTokens']} total")

👤 Query: How many hotels in the database have a swimming pool?

[TRADITIONAL RAG]


Let me search the FAQs for information about hotels with swimming pools!


Tool #2: search_faqs


   📥 Tool Input:
      query: hotels with swimming pool


   📤 Tool Output (truncated):
[hotel-phoenix-002.txt]
# Sunset Peak Lodge

## Hotel Overview
Sunset Peak Lodge is located in Phoenix, AZ, USA. Luxury accommodations with stunning city views

**Guest Rating:** 4.5/5.0
**Total Rooms:** 290

## Contact Information
**Address:** 1052 Central Avenue, Phoenix, AZ 85002
**Phone:** +1-602-555-0128
**Email:** phoenix2@anycompany.com

## Check-in & Check-out

### Check-in Time
Standard check-in time is 3:00 PM. Early check-in is subject to availability and may be requested at the time of booking or by contact...

[hotel-puebla-001.txt]
# Ceramic Tile Grand Hotel

## Hotel Overview
Ceramic Tile Grand Hotel is located in Puebla, Puebla, Mexico. Refined accommodations with modern conveniences

**Guest Rating:** 4.6/5.0
**Total Rooms:** 280

## Contact Information
**Address:** Avenid...


...

 in Mexico").
3. **Contact our reservations team** – They would

 have full access to the complete hotel database to

 give you an accurate count.

Would you like me to search for pool

 availability in a specific city or hotel? 

🏊
💰 Tokens: 7849 in, 679 out, 8528 total


In [8]:
query = "How many hotels in the database have a swimming pool?"

print("[GRAPH-RAG]")
r = graph_agent(query)

if r.metrics:
    usage = r.metrics.accumulated_usage
    print(f"\n💰 Tokens: {usage['inputTokens']} in, {usage['outputTokens']} out, {usage['totalTokens']} total")

print("\n📊 RAG only sees top 3 docs, cannot count | Graph-RAG executes COUNT() across all 300 hotels")

[GRAPH-RAG]


I'll look that up for you right away!


Tool #2: query_knowledge_graph


   📥 Tool Input:
      cypher_query: MATCH (h:Hotel)-[:OFFERS_AMENITY]->(a:Amenity) WHERE toLower(a.name) CONTAINS 'swimming pool' OR toLower(a.name) CONTAINS 'pool' RETURN count(DISTINCT h) AS hotels_with_pool


   📤 Tool Output:
Found 1 results:
  {'hotels_with_pool': 11}



There are **11 hotels** in our database that feature a **swimming pool**! 🏊‍♂

️

Would you like me to list those hotels, filter them by a

 specific city, or get more details about their rates and amenities? I'm here to help you find the perfect stay

!
💰 Tokens: 4060 in, 367 out, 4427 total

📊 RAG only sees top 3 docs, cannot count | Graph-RAG executes COUNT() across all 300 hotels


---

## Test 4: Out-of-Domain Detection

**Paper:** "RAG hallucinates when data doesn't exist — returns plausible but fabricated answers"

**Query:** Tell me about hotels in Antarctica

In [9]:
query = "Tell me about hotels in Antarctica"
print(f"👤 Query: {query}\n")

print("[TRADITIONAL RAG]")
r_rag = rag_agent(query)

if r_rag.metrics:
    usage = r_rag.metrics.accumulated_usage
    print(f"\n💰 Tokens: {usage['inputTokens']} in, {usage['outputTokens']} out, {usage['totalTokens']} total")

👤 Query: Tell me about hotels in Antarctica

[TRADITIONAL RAG]


Let me search the FAQs for information about hotels in Antarctica!
Tool #4: search_faqs


   📥 Tool Input:
      query: hotels in Antarctica


   📤 Tool Output (truncated):
[hotel-anchorage-001.txt]
# Aurora Grand

## Hotel Overview
Aurora Grand is located in Anchorage, AK, USA. Alaskan adventure with urban comfort

**Guest Rating:** 4.6/5.0
**Total Rooms:** 220

## Contact Information
**Address:** 1481 4th Avenue, Anchorage, AK 99501
**Phone:** +1-907-555-0293
**Email:** anchorage1@anycompany.com

## Check-in & Check-out

### Check-in Time
Standard check-in time is 3:00 PM. Early check-in is subject to availability and may be requested at the time of booking or by contacting the hotel dire...

[hotel-anchorage-002.txt]
# Peak View Suites

## Hotel Overview
Peak View Suites is located in Anchorage, AK, USA. Northern luxury with glacier views

**Guest Rating:** 4.7/5.0
**Total Rooms:** 200

## Contact Information
**Address:** 1482 4th Avenue, Anchorage, AK 995...


It appears that there are **no hotels in Antarctica** in our

...

** – Cold climate with modern comforts
- 

🗼 **Other unique destinations** in our

 database

Would you like more details on any of these,

 or is there another destination I can help you explore?


💰 Tokens: 14174 in, 951 out, 15125 total


In [10]:
query = "Which hotels in Cairo have both a spa and a swimming pool, and what are their guest ratings?"
print(f"👤 Query: {query}\n")

print("[TRADITIONAL RAG]")
r = rag_agent(query)

if r.metrics:
    usage = r.metrics.accumulated_usage
    print(f"\n💰 Tokens: {usage['inputTokens']} in, {usage['outputTokens']} out, {usage['totalTokens']} total")

👤 Query: Which hotels in Cairo have both a spa and a swimming pool, and what are their guest ratings?

[TRADITIONAL RAG]


Let me search for both spa and swimming pool information for hotels in Cairo simultaneously!


Tool #5: search_faqs



Tool #6: search_faqs


   📥 Tool Input:
      query: hotels in Cairo with spa
   📥 Tool Input:
      query: hotels in Cairo with swimming pool


...

 filter hotels by specific amenities across the

 full database.

Would you like help

 with anything else about these Cairo hotels or another destination

? 🐪

🏛️
💰 Tokens: 28878 in, 1563 out, 30441 total


In [11]:
query = "Which hotels in Cairo have both a spa and a swimming pool, and what are their guest ratings?"

print("[GRAPH-RAG]")
r = graph_agent(query)

if r.metrics:
    usage = r.metrics.accumulated_usage
    print(f"\n💰 Tokens: {usage['inputTokens']} in, {usage['outputTokens']} out, {usage['totalTokens']} total")

print("\n📊 RAG finds partial matches, cannot filter by multiple criteria | Graph-RAG traverses Hotel→Amenity→Amenity with AND logic")

[GRAPH-RAG]


Let me find that for you right away!


Tool #3: query_knowledge_graph


   📥 Tool Input:
      cypher_query: MATCH (h:Hotel)-[:OFFERS_AMENITY]->(a1:Amenity), (h)-[:OFFERS_AMENITY]->(a2:Amenity) WHERE h.address CONTAINS 'Cairo' AND (toLower(a1.name) CONTAINS 'spa') AND (toLower(a2.name) CONTAINS 'pool') RETURN DISTINCT h.name AS hotel_name, h.guest_rating AS guest_rating ORDER BY guest_rating DESC


   📤 Tool Output:
Found 2 results:
  {'hotel_name': 'AnyCompany Giza Pyramids', 'guest_rating': 4.6}
  {'hotel_name': 'AnyCompany Cairo Nile View', 'guest_rating': 4.5}



...

 5 |

Both hotels are highly rated! The **AnyCompany Giza

 Pyramids** edges ahead with a slightly higher guest rating of 4.6.

 Would you like more details about either of these hotels, such as room types, pricing, or additional

 amenities? I'd be happy to help! 🌍

✨
💰 Tokens: 6928 in, 712 out, 7640 total

📊 RAG finds partial matches, cannot filter by multiple criteria | Graph-RAG traverses Hotel→Amenity→Amenity with AND logic


In [12]:
query = "Tell me about hotels in Antarctica"

print("\n[GRAPH-RAG]")
r_graph = graph_agent(query)

if r_graph.metrics:
    usage = r_graph.metrics.accumulated_usage
    print(f"\n💰 Tokens: {usage['inputTokens']} in, {usage['outputTokens']} out, {usage['totalTokens']} total")

print("\n📊 RAG may hallucinate plausible info | Graph-RAG returns 'No hotels found' (honest failure)")


[GRAPH-RAG]


Let me check the database for you!


Tool #4: query_knowledge_graph


   📥 Tool Input:
      cypher_query: MATCH (h:Hotel) WHERE h.address CONTAINS 'Antarctica' RETURN h


   📤 Tool Output:
No results found.


...

 locations
- 🗼 **European getaways** like

 Paris
- 🌍 Any other destination you have in mind



Just let me know where you'd

 like to explore, and I'll find the best hotel options for you!


💰 Tokens: 10507 in, 962 out, 11469 total

📊 RAG may hallucinate plausible info | Graph-RAG returns 'No hotels found' (honest failure)
